# Lecture Notes: Trouble Sort (Google Code Jam 2018 Qualifiers – Problem A)

This lecture explains the **Trouble Sort** problem from Google Code Jam 2018, and shows how to simulate it efficiently without actually running the slow brute‑force version.

---

## 1. Problem Overview and Motivation

- **Trouble Sort** is a variation of bubble sort, but it operates on **triplets** instead of pairs.  
- The goal is to determine whether Trouble Sort successfully sorts the input array into non‑decreasing order.  
- If the array is **not sorted after** Trouble Sort, we must return the **smallest index** where an adjacent inversion occurs (i.e., the first position where `A[i] > A[i+1]`).

**Design constraints:**
- Arrays can be very large (up to \(10^9\) elements), so simulating the naive Trouble Sort would be too slow.

---

## 2. What Trouble Sort Actually Does

### Bubble Sort Recap

- In **bubble sort**, we repeatedly pass over the array and compare **adjacent pairs**.  
- If `A[i] > A[i+1]`, we swap them until no more swaps are needed.

### Trouble Sort Definition

- In **Trouble Sort**, we look at **consecutive triplets** of elements: `A[i], A[i+1], A[i+2]`.  
- We compare the **first (`A[i]`) and last (`A[i+2]`)** of the triplet.  
- If `A[i] > A[i+2]`, we **reverse the triplet**, which effectively **swaps `A[i]` and `A[i+2]`**, while `A[i+1]` stays in place.

### Pseudocode: Trouble Sort (High‑Level)

```text
Algorithm TroubleSort(L):
    n ← length of L
    done ← false

    while done = false do:
        done ← true
        for i ← 0 to n−3 do:
            if L[i] > L[i+2] then:
                done ← false
                swap L[i] with L[i+2]
    end while
```

**Note:**
- The loop runs from `i = 0` to `n−3` because we need enough space for a triplet starting at `i`.

---

## 3. Why Simulating Trouble Sort Is Too Slow

- The **outer while loop** may run up to \(n\) times in the worst case.  
- The **inner for loop** runs about \(n\) times per pass.  
- Hence, worst‑case time complexity is \(O(n^2)\).

**Given:**
- Input size can be up to \(10^9\).  
- \(O(n^2)\) is **not feasible** within time limits.

We must **avoid** literally simulating Trouble Sort and instead **compute its outcome indirectly**.

---

## 4. Key Insight: Two Independent Bubble Sorts

### Parity of Indices

- Notice that Trouble Sort **only ever compares or swaps elements at positions with the same parity**:
  - Even indices: `0, 2, 4, ...`
  - Odd indices:  `1, 3, 5, ...`  
- Specifically:
  - In the first pass, `i` takes values `0, 1, 2, 3, ...`, so:
    - `L[0] ↔ L[2]` (even ↔ even)
    - `L[1] ↔ L[3]` (odd ↔ odd)
    - `L[2] ↔ L[4]` (even ↔ even)
    - `L[3] ↔ L[5]` (odd ↔ odd)
  - So:
    - **Even‑indexed elements** are only compared/swapped among themselves.
    - **Odd‑indexed elements** are only compared/swapped among themselves.

### Trouble Sort = Two Parallel Bubble Sorts

- **Trouble Sort is equivalent to:**
  - Running **bubble sort on the sub‑array of even‑indexed elements**.
  - Running **bubble sort on the sub‑array of odd‑indexed elements**.  
- After these two independent sorts, the final array is formed by:
  - Placing the sorted even‑indexed elements back into even positions.
  - Placing the sorted odd‑indexed elements back into odd positions.

So:
- We can **simulate Trouble Sort efficiently** as:
  - Split the array into two sub‑arrays by index parity.
  - Sort each sub‑array with an efficient \(O(n \log n)\) sort (e.g., built‑in sort).
  - Merge them back to form the Trouble Sort result.

---

## 5. Efficient Algorithm (Pseudocode)

### Step 1: Split Even and Odd Sub‑Arrays

```text
Input: Array A of length n

Initialize:
    evenList ← empty list
    oddList  ← empty list

for i ← 0 to n−1 do:
    if i is even:
        append A[i] to evenList
    else:
        append A[i] to oddList
```

### Step 2: Sort Each Sub‑Array Efficiently

```text
sort(evenList)   // using O(|evenList| log |evenList|) sort
sort(oddList)    // using O(|oddList| log |oddList|) sort
```

### Step 3: Merge Back into Trouble Sort Output

```text
Initialize:
    output ← array of length n

Initialize pointers:
    evenIndex ← 0
    oddIndex  ← 0

for i ← 0 to n−1 do:
    if i is even:
        output[i] ← evenList[evenIndex]
        evenIndex ← evenIndex + 1
    else:
        output[i] ← oddList[oddIndex]
        oddIndex  ← oddIndex + 1
```

Now, `output` is the **array that Trouble Sort would produce**.

### Step 4: Check If Sorted and Find First Inversion

```text
Set flag ← −1   // will store first faulty index

for i ← 0 to n−2 do:
    if output[i] > output[i+1] then:
        flag ← i
        break out of loop

if flag = −1:
    print "OK"
else:
    print flag   // smallest index where Trouble Sort fails
```

### Why This Works

- The **even‑indexed** and **odd‑indexed** elements are each sorted independently, so the structure of Trouble Sort is preserved.  
- The final merge completely reconstructs the Trouble‑sorted array.  
- Then a single pass checks if the full array is non‑decreasing, and finds the first inversion if not.

**Time complexity:**
- Splitting: \(O(n)\).
- Two sorts: \(O(n \log n)\).
- Merging and checking: \(O(n)\).  
Overall: \(O(n \log n)\), which is feasible for large inputs.

---

## 6. Why Trouble Sort Is Not Always Correct

Consider the example given in the lecture:

- Input: `[1, 3, 2]`  
- Indices: `0, 1, 2`  
- Even indices: positions `0, 2` → values `[1, 2]` (already sorted).  
- Odd indices: position `1` → value `[3]` (trivially sorted).  
- After Trouble Sort:
  - Even‑indexed part: `[1, 2]`.
  - Odd‑indexed part: `[3]`.
  - Final array: `[1, 3, 2]` → **not sorted**, because `3 > 2`.  
- First inversion appears at index `1` (i.e., `A[1] > A[2]`).

So:
- Trouble Sort can **fail** even on small arrays.
- The problem asks us to **detect** such failures and report the first faulty index.

---

## 7. Correctness of \(O(n^2)\) Bound

- Since Trouble Sort is equivalent to two parallel bubble sorts (on even and odd sub‑arrays), its behavior mirrors bubble sort on each sub‑array.  
- On each sub‑array, bubble sort takes at most \(k^2\) time for size \(k\).  
- The outer while loop runs at most \(n\) times because:
  - Each pass can improve the order of the even‑indexed and odd‑indexed elements only so much.
  - After a finite number of passes (at most \(n\)), no more triplets violate the order.

Thus, the **worst‑case time if you simulate Trouble Sort** is \(O(n^2)\), but this is not acceptable for large inputs. Hence the need for the efficient reconstruction method above.

---

## 8. Summary of Key Takeaways

- **Trouble Sort**:
  - Operates on triplets, comparing `A[i]` and `A[i+2]`.  
  - If `A[i] > A[i+2]`, it swaps them (middle element stays in place).

- **What it effectively does**:
  - Runs **bubble‑like sorting independently** on even‑indexed and odd‑indexed elements.

- **Efficient solution strategy**:
  1. Split the array into **even‑indexed** and **odd‑indexed** sub‑arrays.
  2. Sort each sub‑array with an efficient \(O(n \log n)\) algorithm.
  3. Merge them back to form the Trouble‑sorted array.
  4. Check if the merged array is non‑decreasing, and find the **first index where `A[i] > A[i+1]`** if not.

- **Why this is important**:
  - Direct simulation is \(O(n^2)\), which is too slow for \(n \approx 10^9\).  
  - The parity‑based reconstruction runs in \(O(n \log n)\) and is the intended solution for this problem.

This approach turns a tricky “simulate and watch” problem into a **clean, analytical insight** about index parity and sorting, which is exactly the style of reasoning valued in competitive programming.